In [1]:
%load_ext autoreload
%autoreload 2

import mag_dephygraph as mag

import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime

In [2]:
import geopandas as gpd

ENTREPOT_PATH = '/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/'
GEOVEC_PATH = '/home/administrateur/Bureau/Datagrosyst/catalogue_script_agrosyst/02_outils/data/external_data/geospatial_data/'

donnees = {}

def import_df(df_name, path_data, sep, file_format='csv') :
    """
        importe un dataframe au chemin path_data+df_name+'.csv' et le stock dans le dictionnaire 'df' à la clé df_name
    """
    global donnees
    if file_format == 'csv' :
        donnees[df_name] = pd.read_csv(path_data+df_name+'.'+file_format, sep = sep, low_memory=False).replace({'\r\n': '\n'}, regex=True)
    if file_format == 'json' and df_name.startswith('geoVec') :
        # Utilise geopandas pour les json formater en geojson. Le nom du fichier json doit alors commencer par geoVec
        donnees[df_name] = gpd.read_file(path_data+df_name+'.'+file_format)
    if file_format == 'gpkg' :
        donnees[df_name] = gpd.read_file(path_data+df_name+'.'+file_format)

def import_dfs(df_names, path_data, sep = ',', verbose=False, file_format='csv') :
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, file_format)

tables = [
    "sdc_realise_performance",
    "synthetise_synthetise_performance",
    "synthetise",
    "sdc",
    "dispositif",
    "domaine",
    "commune",
    "identification_pz0",
    "espece_variete_perenne_principale",
    "entite_unique_par_sdc_nettoyage",
    "itk_realise_agrege",
]

# import des données du magasin
import_dfs(tables, ENTREPOT_PATH, sep = ',')

import_dfs(["geoVec_arrond","geoVec_dep"], GEOVEC_PATH, sep = ',', file_format='json')

100%|██████████| 2/2 [00:00<00:00,  5.80it/s]


In [3]:
GCPE = ['GRANDES_CULTURES','POLYCULTURE_ELEVAGE']

PERFORMANCES_COLS = [
    'approche_de_calcul',
    'ift_cible_non_mil_tx_comp',
    # IFT
    'ift_cible_non_mil_chimique_tot',
    'ift_cible_non_mil_chim_tot_hts',
    'ift_cible_non_mil_h',
    'ift_cible_non_mil_f',
    'ift_cible_non_mil_i',
    'ift_cible_non_mil_ts',
    'ift_cible_non_mil_a',
    'ift_cible_non_mil_hh',
    'ift_cible_non_mil_biocontrole',
    'recours_aux_moyens_biologiques',
    # Temps
    'tps_utilisation_materiel',
    'tps_travail_manuel',
    'tps_travail_total',
    # UTH
    'surface_par_unite_de_travail_humain', # noneed
    'nombre_uth_necessaires', # noneed
    # Travail du sol
    'nbre_de_passages_desherbage_meca',
    'utili_desherbage_meca',
    'type_de_travail_du_sol',
    # Charges
    'co_tot_std_mil',
    'cm_std_mil',
    'c_main_oeuvre_tot_std_mil',
    'c_main_oeuvre_tractoriste_std_mil', # noneed
    'c_main_oeuvre_manuelle_std_mil', # noneed
    # Consommation
    'conso_carburant',
    # Economique
    'pb_std_mil_avec_autoconso',
    'mb_std_mil_avec_autoconso',
    'msn_reelle_avec_autoconso',
    'md_std_mil_avec_autoconso', # noneed
    # Fertilistation
    'ferti_n_tot',
    'ferti_n_mineral',
    'ferti_n_organique',
    'ferti_p2o5_tot',
    'ferti_p2o5_mineral',
    'ferti_p2o5_organique',
    'ferti_k2o_tot',
    'ferti_k2o_mineral',
    'ferti_k2o_organique',
    # HRI1
    'hri1_hts',
    'hri1_g1_hts',
    'hri1_g2_hts',
    'hri1_g3_hts',
    'hri1_g4_hts',
    # QSA
    'qsa_tot_hts',
    'qsa_danger_environnement_hts',
    'qsa_toxique_utilisateur_hts',
    'qsa_cmr_hts',
    'qsa_glyphosate_hts',
    'qsa_cuivre_tot_hts',
    'qsa_soufre_tot_hts'
    ]

ALERTES_COLS_VAR = {
    'alerte_ferti_n_tot' : "ferti_n_tot",
    'alerte_ift_cible_non_mil_chim_tot_hts' : "ift_cible_non_mil_chim_tot_hts",
    'alerte_ift_cible_non_mil_f': "ift_cible_non_mil_f",
    'alerte_ift_cible_non_mil_h' : "ift_cible_non_mil_h",
    'alerte_ift_cible_non_mil_i' : "ift_cible_non_mil_i",
    'alerte_ift_cible_non_mil_biocontrole' : "ift_cible_non_mil_biocontrole",
    'alerte_co_irrigation_std_mil' : "co_irrigation_std_mil",
    'alerte_msn_std_mil_avec_autoconso' : "msn_std_mil_avec_autoconso",
    # 'alerte_nombre_interventions_phyto' : ["???"],
    'alerte_pb_std_mil_avec_autoconso' : "pb_std_mil_avec_autoconso",
    # 'alerte_rendement' : ['pb_std_mil_avec_autoconso','mb_std_mil_avec_autoconso', 'msn_reelle_avec_autoconso', 'md_std_mil_avec_autoconso'],
    'alertes_charges' : ["co_tot_std_mil","cm_std_mil"],
    'alerte_cm_std_mil' : "cm_std_mil",
    'alerte_co_semis_std_mil' : "co_semis_std_mil"
}

In [4]:
# Chargement des données nécessaires
    ## Performances
sdc_realise_performance = donnees["sdc_realise_performance"][['sdc_id'] + PERFORMANCES_COLS + list(ALERTES_COLS_VAR.keys())]
synthetise_synthetise_performance = donnees["synthetise_synthetise_performance"][['synthetise_id'] + PERFORMANCES_COLS + list(ALERTES_COLS_VAR.keys())]
    ## Entrepot
synthetise = donnees["synthetise"][['id', 'nom', 'campagnes', 'sdc_id']].rename(columns={'id':'synthetise_id', 'campagnes':'synthetise_campagne'})
sdc = donnees["sdc"][['id','code','nom','modalite_suivi_dephy','code_dephy','filiere','type_production','type_agriculture','part_sau_domaine','reseaux_ir','reseaux_it','dispositif_id','validite']].rename(columns={"id": "sdc_id"})
dispositif = donnees["dispositif"][['id','type','domaine_id']].rename(columns={"id": "dispositif_id"})
domaine = donnees["domaine"][['id','code','nom','campagne','commune_id','sau_totale']].rename(columns={"id": "domaine_id"})
commune = donnees["commune"][['id','codeinsee', 'departement', 'region', 'bassin_viticole', 'ancienne_region','latitude', 'longitude','arrondissement_code']].rename(columns={"id": "commune_id"})
    ## Referentiel
arrond_data = donnees['geoVec_arrond']
dep_data = donnees['geoVec_dep']
    ## Outils
identification_pz0 = donnees["identification_pz0"]
espece_variete_perenne_principale = donnees['espece_variete_perenne_principale'][['entite_id','espece_principale','variete_principale']]
entite_unique_par_sdc_nettoyage = donnees['entite_unique_par_sdc_nettoyage']
itk_realise_agrege = donnees['itk_realise_agrege'][['sdc_id','zone_id']]

In [5]:
# Construction du jeu de données de base pour le magasin DEPHYGraph
    ## Création des S à partir de l'outil d'entité unique par SDC
synth = pd.merge(
    entite_unique_par_sdc_nettoyage.loc[entite_unique_par_sdc_nettoyage["entite_retenue"] != "realise_retenu"].rename(columns={"entite_retenue": "synthetise_id"}), 
    synthetise_synthetise_performance, on="synthetise_id", how="left"
)
synth = synth.merge(synthetise.drop(columns='sdc_id').rename(columns={"nom": "nom_synthetise"}), on='synthetise_id', how='left', suffixes=(None,'_synthetise'))
    ## Création des R à partir de l'outil d'entité unique par SDC
realise = pd.merge(
    entite_unique_par_sdc_nettoyage.loc[entite_unique_par_sdc_nettoyage["entite_retenue"] == "realise_retenu"], sdc_realise_performance, on="sdc_id", how="left"
)
    ## Concaténation S + R
df = pd.concat([synth, realise])
    ## Ajout des infos sdc, dispositif et domaine
df = df.merge(sdc, on='sdc_id', how='left', suffixes=(None,'_sdc'))
df = df.merge(dispositif, on='dispositif_id', how='left', suffixes=(None,'_dispositif'))
df = df.merge(domaine, on='domaine_id', how='left', suffixes=(None,'_domaine'))
df['entite_id'] = np.where(df['synthetise_id'].notna(), df['synthetise_id'], df['sdc_id'])

In [6]:
df['code_dephy_nettoyage'] = mag.nettoyage_numero_dephy(df, colonne='code_dephy')
df['code_dephy_nettoyage'] = np.where(df['code_dephy_nettoyage'].isna(), df['code_dephy'], df['code_dephy_nettoyage'])

In [7]:
# Filtre des dispositifs Ferme Detaille avec un type d'agriculture renseigné
df = mag.filtre_1_dispoFermeDetaille_typeAgriNotNull(df)

Filtre 1 : -7195 lignes


In [8]:
# Filtre des années trop vieilles ou trop récentes
df = mag.filtre_2_annees(df)

Filtre 2 : -7461 lignes


In [9]:
# Identification des pz0 et filtrage des entités n'étant pas un état valide (pz0 ou post)
df = mag.identificationDesPz0_et_filtrePz0Detecte(df, itk_realise_agrege, identification_pz0)

Identification pz0 : -3470 lignes


In [ ]:
# On explode les campagnes pluri-annuelles,
df = mag.explode_campagne(df)

Explode campagne : -1328 lignes


In [11]:
# Modification des duplicats de numéro DEPHY*campagne
df = mag.modification_duplicat_codeDephy_newCampagne(df)

Modification de 40 code_dephy en duplication sur la même campagne. 15 code_dephy différents concernés.


In [ ]:
df = mag.unique_pz0_by_code_dephy(df)

In [12]:
# Ajout des infos sur les espèces et variétés principales pour les cultures pérennes
df = df.merge(espece_variete_perenne_principale, on = 'entite_id', how='left')
df = mag.ajout_especes_et_varietes_principales(df)

In [13]:
# Ajout des typologie de systèmes simplifiées
df = mag.ajout_typologie_simplifiee(df)

c120_arboriculture_typo_sdc :
 [None 'Abricotiers et Pruniers conventionnels'
 'Pommiers et Poiriers conventionnels' 'Pommiers et Poiriers biologiques'
 'Oliviers conventionnels' 'Clémentiniers conventionnels'
 'Pêchers biologiques' 'Pêchers conventionnels'
 'Abricotiers et Pruniers biologiques' 'Oliviers biologiques'
 'Noyers biologiques' 'Noyers conventionnels' 'Clémentiniers biologiques'
 'Cerisiers conventionnels'] 

c121_maraichage_typo_sdc :
 [None 'Abris conventionnels' 'Plein-champ conventionnels'
 'Abris biologiques' 'Plein-champ biologiques' 'Hors-sol conventionnels'] 

c122_horticulture_typo_sdc :
 [None 'Plantes en pot' 'Pépinière hors sol, en conteneur'
 'Pépinières pleine terre' 'Pépinières mixte'] 

c123_cult_tropicales_typo_sdc :
 [None] 

c124_gcpe_typo_sdc :
 [None 'Grandes cultures conventionnels'
 'Polyculture élevage conventionnels' 'Polyculture élevage biologiques'
 'Grandes cultures biologiques'] 



In [14]:
# Ajout de variables supplémentaires
    ## IFT hors herbicide et hors TS
df['c602_IFT_hh_hts'] = df['ift_cible_non_mil_hh'] - df['ift_cible_non_mil_ts']
    ## Charges totales hors main d'oeuvre
df['c504_outLabourTotalExpenses'] = df['co_tot_std_mil'] + df['cm_std_mil']

In [15]:
# Ajout des infos géographiques
df = mag.ajout_infos_geo(df, commune, arrond_data, dep_data)

In [16]:
df['realized'] = np.where(df['synthetise_id'].isna(), 'realise', 'synthetise')

In [17]:
# Filtre de cohérence (Espèces principales en pérennes, Semis direct en AB ou avec IFT faible)
df = mag.filtre_3_coherence(df)

Filtre 3 : -6 lignes ; -0.003% des valeurs


In [18]:
# On supprime les code_dephy n'ayant qu'une seule campagne ou n'ayant aucun pz0 identifié
df = mag.filtre_4_avecUnPz0_et_auMoinsDeuxCampagnes(df)

Filtre 4 : -1146 lignes


In [19]:
# Filtre sur la disponibilité de certaines variables selon les filières
df = mag.filtre_5_disponibilite_par_filiere(df)

Filtre 5 : 10.8% des valeurs


In [20]:
DICT_VAR_IMPACTED = {
    'nbre_de_passages_desherbage_meca': 
				[None],
	'tps_travail_total': 
				[None],
	'tps_utilisation_materiel': 
				['tps_travail_total'],
	'tps_travail_manuel': 
				['tps_travail_total'],
	'conso_carburant': 
				[None],
                

	'ferti_n_tot': 
				[None],
	'ferti_n_mineral': 
				['ferti_n_tot'],
	'ferti_n_organique': 
				['ferti_n_tot'],
	'ferti_p2o5_tot': 
				[None],
	'ferti_p2o5_mineral': 
				['ferti_p2o5_tot'],
	'ferti_p2o5_organique': 
				['ferti_p2o5_tot'],
	'ferti_k2o_tot': 
				[None],
	'ferti_k2o_mineral': 
				['ferti_k2o_tot'],
	'ferti_k2o_organique': 
				['ferti_k2o_tot'],
                

	'pb_std_mil_avec_autoconso': 
				['mb_std_mil_avec_autoconso', 'msn_reelle_avec_autoconso', 'md_std_mil_avec_autoconso'],
	'mb_std_mil_avec_autoconso': 
				['msn_reelle_avec_autoconso', 'c504_outLabourTotalExpenses', 'md_std_mil_avec_autoconso'],
	'msn_reelle_avec_autoconso': 
    			[None],
	'c504_outLabourTotalExpenses': 
				['msn_reelle_avec_autoconso','md_std_mil_avec_autoconso'],
	'co_tot_std_mil': 
				['mb_std_mil_avec_autoconso', 'msn_reelle_avec_autoconso', 'c504_outLabourTotalExpenses','md_std_mil_avec_autoconso'],
	'c_main_oeuvre_tot_std_mil': 
				['md_std_mil_avec_autoconso'],
	'cm_std_mil': 
				['msn_reelle_avec_autoconso', 'c504_outLabourTotalExpenses','md_std_mil_avec_autoconso'],
    'md_std_mil_avec_autoconso':
				[None], # si les indicateur éco sont mauvais, on passe la marge directe en NA. L'inverse, on verra plus tard, quand la marge directe sera validée par la CAN                


	'ift_cible_non_mil_chim_tot_hts': 
				['hri1_hts','hri1_g1_hts','hri1_g2_hts','hri1_g3_hts','hri1_g4_hts',
     			'qsa_tot_hts'],
	'c602_IFT_hh_hts': 
				['hri1_hts','hri1_g1_hts','hri1_g2_hts','hri1_g3_hts','hri1_g4_hts',
     			'ift_cible_non_mil_chim_tot_hts', 'qsa_tot_hts'],
	'ift_cible_non_mil_biocontrole': 
				['hri1_hts','hri1_g1_hts','hri1_g2_hts','hri1_g3_hts','hri1_g4_hts',
     			'qsa_tot_hts'],
	'ift_cible_non_mil_h': 
				['hri1_hts','hri1_g1_hts','hri1_g2_hts','hri1_g3_hts','hri1_g4_hts',
     			'ift_cible_non_mil_chim_tot_hts', 'qsa_tot_hts'],
	'ift_cible_non_mil_i': 
				['hri1_hts','hri1_g1_hts','hri1_g2_hts','hri1_g3_hts','hri1_g4_hts',
     			'ift_cible_non_mil_chim_tot_hts', 'c602_IFT_hh_hts', 'qsa_tot_hts'],
	'ift_cible_non_mil_f': 
				['hri1_hts','hri1_g1_hts','hri1_g2_hts','hri1_g3_hts','hri1_g4_hts',
     			'ift_cible_non_mil_chim_tot_hts', 'c602_IFT_hh_hts', 'qsa_tot_hts'],
	'ift_cible_non_mil_a': 
				['hri1_hts','hri1_g1_hts','hri1_g2_hts','hri1_g3_hts','hri1_g4_hts',
     			'ift_cible_non_mil_chim_tot_hts', 'c602_IFT_hh_hts', 'qsa_tot_hts'],
	'recours_aux_moyens_biologiques': 
				[None],
                

	'hri1_hts': 
				['hri1_g1_hts','hri1_g2_hts','hri1_g3_hts','hri1_g4_hts'],
	'hri1_g1_hts': 
				['hri1_hts','hri1_g2_hts','hri1_g3_hts','hri1_g4_hts'],
	'hri1_g2_hts': 
				['hri1_g1_hts','hri1_hts','hri1_g3_hts','hri1_g4_hts'],
	'hri1_g3_hts': 
				['hri1_g1_hts','hri1_g2_hts','hri1_hts','hri1_g4_hts'],
	'hri1_g4_hts': 
				['hri1_g1_hts','hri1_g2_hts','hri1_g3_hts','hri1_hts'],
    

	'qsa_tot_hts': 
				[None],
                # ['hri1_hts', 'ift_cible_non_mil_chim_tot_hts'],
	'qsa_cmr_hts': 
				['qsa_tot_hts'],
	'qsa_toxique_utilisateur_hts': 
				['qsa_tot_hts'],
	'qsa_glyphosate_hts': 
				['qsa_tot_hts'],
	'qsa_danger_environnement_hts': 
				['qsa_tot_hts'],
	'qsa_cuivre_tot_hts': 
				['qsa_tot_hts'],
	'qsa_soufre_tot_hts': 
				['qsa_tot_hts']
}

In [21]:
# for x in list(ALERTES_COLS_VAR.keys()):
#     print(
#         x," : ",
#         len(df.loc[~df[x].isin(
#             ["Pas d'alerte","Cette alerte n'existe pas encore dans cette filière"]
#             ), x])
#     )

In [22]:
# On crée un index code_dephy * new_campagne
if df.duplicated(['code_dephy',"new_campagne"]).any():
    raise ValueError("code_dephy * new_campagne n'est pas unique, pas d'index possible !")

df["new_campagne_str"] = df["new_campagne"].astype(str)
df["code_dephy_for_idx"] = df["code_dephy"].astype(str)

df.set_index(['code_dephy_for_idx',"new_campagne_str"], inplace=True)

In [23]:
# def detect_outliers_mad(df, dict_var_impacted, thresh=3.0):
#     dict_index_outliers = {}
#     for col in dict_var_impacted:
#         s = df[col]
#         med = s.median()
#         mad = (s - med).abs().median()
#         z = (s - med) / (1.4826 * mad)
#         dict_index_outliers[col] = s[z.abs() > thresh].index.tolist()
#     return dict_index_outliers

In [24]:
dict_idx_iqr = mag.detect_outliers_via_iqr(df, DICT_VAR_IMPACTED, coef=2)
df = mag.apply_nan(df, DICT_VAR_IMPACTED, dict_idx_iqr)

# dict_idx_mad = detect_outliers_mad(df, dict_var_impacted, thresh=3.0)
# df_mad = apply_nan(df, dict_var_impacted, dict_idx_mad)

Filtre outlier : 4.7% des valeurs (soit 78618 valeurs)


In [25]:
# from ydata_profiling import ProfileReport
# # profile_df = ProfileReport(df[list(dict_var_impacted.keys())], title="DF rapport")
# # profile_df.to_file("distrib_df.html")

# profile_iqr = ProfileReport(df[list(DICT_VAR_IMPACTED.keys())], title="IQR rapport")
# profile_iqr.to_file("distrib_df_iqr.html")

# # profile_mad = ProfileReport(df_mad[list(dict_var_impacted.keys())], title="MAD rapport")
# # profile_mad.to_file("distrib_df_mad.html")

# # Le mieux = IQR, voir quel coef est le mieux

In [26]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# def compute_distances(df, dict_var_impacted, coef):
#     dists = []
#     for col in dict_var_impacted:
#         s = df[col]
#         q1, q3 = s.quantile([0.25, 0.75])
#         iqr = q3 - q1
#         low, high = q1 - coef * iqr, q3 + coef * iqr

#         dist = (low - s).clip(lower=0) + (s - high).clip(lower=0)
#         dists.append(dist)

#     return pd.concat(dists)


# def summary_stats(df, dict_var_impacted, coef):
#     d = compute_distances(df, dict_var_impacted, coef)
#     d = d[d > 0]

#     return {
#         "coef": coef,
#         "n_outliers": len(d),
#         "mean_dist": d.mean() if len(d) else 0,
#         "median_dist": d.median() if len(d) else 0
#     }


# coefs = np.linspace(0.5, 5, 20)
# results = pd.DataFrame([summary_stats(df, dict_var_impacted, c) for c in coefs])

# # --- PLOT DISTANCES (robuste) ---
# plt.figure()

# for c in [1, 1.5, 2, 2.5, 3]:
#     d = compute_distances(df, dict_var_impacted, c)
#     d = d[d > 0]

#     if len(d):
#         # clip pour enlever les extrêmes (ex: 99e percentile)
#         cap = d.quantile(0.99)
#         d_clip = d.clip(upper=cap)

#         # log pour compresser l’échelle
#         d_log = np.log1p(d_clip)

#         plt.hist(d_log, bins=50, alpha=0.4, label=f"coef={c}", density=True)

# plt.xlabel("log(1 + distance)")
# plt.ylabel("densité")
# plt.title("Distribution des distances (clippée + log)")
# plt.legend()
# plt.savefig("iqr_distances.png", dpi=150)

# # --- PLOT HEURISTIQUE ---
# plt.figure()
# plt.plot(results["coef"], results["n_outliers"], label="n_outliers")
# plt.plot(results["coef"], results["mean_dist"], label="mean_dist")
# plt.xlabel("coef")
# plt.title("Heuristique choix coef")
# plt.legend()
# plt.savefig("iqr_heuristique.png", dpi=150)

# results.head()

In [27]:
dict_idx_alerte_can = mag.detect_outliers_via_alerte_can(df, ALERTES_COLS_VAR)
df = mag.apply_nan(df, DICT_VAR_IMPACTED, dict_idx_alerte_can)

Filtre outlier : 0.7% des valeurs (soit 11482 valeurs)


In [28]:
# from ydata_profiling import ProfileReport
# # profile_df = ProfileReport(df[list(dict_var_impacted.keys())], title="DF rapport")
# # profile_df.to_file("distrib_df.html")

# profile_iqr = ProfileReport(df[list(DICT_VAR_IMPACTED.keys())], title="IQR rapport")
# profile_iqr.to_file("distrib_df_iqr.html")

In [ ]:
df = df.groupby('code_dephy').apply(mag.apply_evol).reset_index(drop=True)

ValueError: Groupe dephy avec trop de pz0. Ils aurait dû être filtrés pour n'en avoir qu'un seul

In [ ]:
df